In [1]:
%pip install langchain_community pypdf langchain_huggingface langchain_chroma langchain_experimental langgraph langchain_fireworks
import os
import warnings
import asyncio
import logging
import gradio as gr
import tkinter as tk
from dotenv import dotenv_values, load_dotenv
from functools import partial
from tkinter import filedialog
from typing_extensions import List, Literal, TypedDict, Dict, Union, Type, Generator, AsyncGenerator


from langsmith import Client

from langchain import hub
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
from langchain.schema import HumanMessage, AIMessage
from langchain.schema.runnable import RunnablePassthrough
from langchain_core.runnables import RunnableLambda, Runnable
from langchain.chat_models import init_chat_model
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_community.document_loaders.text import TextLoader
from langchain_community.document_loaders import PyPDFLoader
from langchain_experimental.text_splitter import SemanticChunker
from langchain_text_splitters.base import TextSplitter
from langchain_core.documents.base import Document
from langchain_core.messages import BaseMessage
from langchain_core.documents.transformers import BaseDocumentTransformer

from langgraph.graph import StateGraph, START

logger_ = logging.getLogger(__name__)
# Create console handler
console_handler = logging.StreamHandler()
console_handler.setLevel(logging.INFO)

# Add the handler to your logger
logger_.addHandler(console_handler)
logger_.setLevel(logging.INFO)

  Using cached onnxruntime-1.22.0-cp312-cp312-win_amd64.whl.metadata (5.0 kB)
Using cached onnxruntime-1.22.0-cp312-cp312-win_amd64.whl (12.7 MB)
Note: you may need to restart the kernel to use updated packages.


c:\Users\Theo Akbas\anaconda3\envs\moodle-assistant\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()


def lc_client_init(env_config=dotenv_values()):
    """Initialize langchain client with provided environment configuration."""
    try:
        _LANGCHAIN_API_KEY = env_config.get("LANGCHAIN_API_KEY")
        if not _LANGCHAIN_API_KEY:
            print("No API key found")
        client = Client(api_key=_LANGCHAIN_API_KEY)
        return client
    except Exception as e:
        print(e)


client = lc_client_init()
print(client)

Client (API URL: https://api.smith.langchain.com)


In [3]:
def pull_prompt(client, prompt_url="rlm/rag-prompt", include_model=True):
    """Pull prompt from LangChain Hub."""
    if not client:
        return None
    try:
        prompt_template = hub.pull(prompt_url, include_model=include_model)
        return prompt_template
    except Exception as e:
        return None


prompt = pull_prompt(client, include_model=False)
prompt_model = pull_prompt(client)
print(f"With model:\n{prompt_model}")
print(f"\nWithout model:\n{prompt}")  # same thing

With model:
input_variables=['context', 'question'] input_types={} partial_variables={} metadata={'lc_hub_owner': 'rlm', 'lc_hub_repo': 'rag-prompt', 'lc_hub_commit_hash': '50442af133e61576e74536c6556cefe1fac147cad032f4377b60c436e6cdcb6e'} messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"), additional_kwargs={})]

Without model:
input_variables=['context', 'question'] input_types={} partial_variables={} metadata={'lc_hub_owner': 'rlm', 'lc_hub_repo': 'rag-prompt', 'lc_hub_commit_hash': '50442af133e61576e74536c6556cefe1fac147cad032f4377b60c436e6cdcb6e'} messages=[HumanMessagePromptTemplate(prompt=PromptT

In [4]:
def chat_model_init(
    model_url="accounts/fireworks/models/llama-v3p1-70b-instruct",
    model_provider="fireworks",
):
    """Initialize chat model from specified provider."""
    try:
        model = init_chat_model(model_url, model_provider=model_provider)
        return model
    except Exception as e:
        print("Problem", e)

model = chat_model_init()
print(model.invoke("Hey there").content)

It's nice to meet you. Is there something I can help you with or would you like to chat?


In [5]:
async def generate(state, *, prompt=None, model) -> AsyncGenerator:
    docs_content = "\n\n".join(doc.page_content for doc in state["context"])
    if model and prompt:
        message = prompt.invoke(
            {"question": state["question"], "context": docs_content}
        )
        response = model.invoke(message)
        yield {"answer": response.content}
    else:
        raise

In [8]:
dir_path = "./documents"
all_splits = []
for file in os.listdir(path=dir_path):
    file_path = os.path.join(dir_path, file)
    print("File Path: ", file_path)
    loader = PyPDFLoader(file_path=file_path)
    splits = loader.load()
    print(f"\nSplit: {splits} of length {len(splits)}\n")
    all_splits.extend(splits)  # Use extend instead of append to flatten the list
print(all_splits)

File Path:  ./documents\Build a Retrieval Augmented Generation.pdf

Split: [Document(metadata={'producer': 'Skia/PDF m136', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/136.0.0.0 Safari/537.36', 'creationdate': '2025-05-19T13:33:22+00:00', 'title': 'Build a Retrieval Augmented Generation (RAG) App: Part 1 | 🦜️🔗 LangChain', 'moddate': '2025-05-19T13:33:22+00:00', 'source': './documents\\Build a Retrieval Augmented Generation.pdf', 'total_pages': 32, 'page': 0, 'page_label': '1'}, page_content="Open in Colab Open in Colab\nOpen on GitHub Open on GitHubTutorialsBuild a Retrieval Augmented Generation (RAG) App: Part 1\nBuild a Retrieval\nAugmented Generation (RAG)\nApp: Part 1\nOne of the most powerful applications enabled by LLMs is sophisticated question-answering\n(Q&A) chatbots. These are applications that can answer questions about speciﬁc source\ninformation. These applications use a technique known as Retrieval Augmented Generat

# Using the Chroma Database

In [41]:
embeddings = HuggingFaceEmbeddings(
                model_name="sentence-transformers/all-mpnet-base-v2"
            )

In [42]:
from datetime import datetime

def remove_documents(vector_store, file_paths:Union[List[str], Literal["all"]]="all"):
    """Remove documents from the vector store based on file path.
    
    Args:
        file_paths: List of file paths to remove. If None, clear the entire collection.
    """
    try:
        if file_paths == "all":
            # Clear the entire collection
            vector_store.delete_collection()
            vector_store = Chroma(
                collection_name=vector_store._collection_name,
                embedding_function=embeddings,
                persist_directory=vector_store._persist_directory,
            )
            logger_.info("Cleared entire vector store collection")
            return True
            
        # For specific files, we need their document IDs
        # This requires tracking document metadata during insertion
        ids_to_remove = []
        for file_path in file_paths:
            # Ensure file_path is a Windows-style path (backslashes)
            file_path = file_path.replace("/", "\\")
            logger_.info(f"File path: {file_path}\n")
            # Get IDs of documents with this file path in metadata
            results = vector_store.get(
                where={"source": file_path}
            )
            logger_.info(f"{datetime.now().strftime("%Y-%m-%d")}   {"INFO"}   {__name__}:   Found a result: {results}")
            if results and "ids" in results:
                logger_.info("Found an ID for the result")
                ids_to_remove.extend(results["ids"])
        
        if ids_to_remove:
            vector_store.delete(ids=ids_to_remove)
            logger_.info(f"Removed {len(ids_to_remove)} documents from vector store")
            return True
        else:
            logger_.info(f"No documents found to remove for the specified file paths")
            return False
            
    except Exception as e:
        logger_.error(f"Failed to remove documents: {str(e)}")
        return False

In [43]:
vector_store = Chroma(
    collection_name="example_collection",
    embedding_function=embeddings,
    persist_directory="./chroma_langchain_db"
)

In [46]:
# Step 2: Investigate what's currently in your database
print("=== Current Database Contents ===")
all_docs = vector_store.get()
print(type(all_docs))
print(f"All docs: {all_docs}")
print(f"All Documents keys: {all_docs.keys()}")
print(f"Total documents: {len(all_docs['ids'])}")
print(f"Document IDs: {all_docs['ids']}")

# Print a sample document from the database
if all_docs["documents"]:
    print(f"\nSample document: {all_docs['documents'][0][:500]}")
else:
    print("\nNo documents found in the database.")

=== Current Database Contents ===
<class 'dict'>
All docs: {'ids': ['98df6b87-62e0-47b9-80b7-935c6d0968cd', '37c7198d-dcf8-4dec-8bd7-e0cb3bcd09f0', 'fe247e61-6868-4d33-9d1b-b9d73b89f021', '5602fbc0-2b8e-44c4-a24b-0a1e2e43617c', 'fb5beb19-5dfd-4920-84db-b6e9e549fdf1', 'b725ba6a-a9b1-454d-921e-4d6d27545b73', 'a68f390b-f605-41bd-a293-9a83041e9307', '2e427473-f651-40c5-9525-0a34aba59ca5', '222ae1ea-d569-4aa9-abf7-acaf1d829e8d', '5d180d3e-d34f-438a-98ae-37aee31ca55a', '8c381208-db85-460c-a020-55c74d87ba0d', '9551ea41-480b-4436-b11f-772807d3fdaa', 'f833e9b0-9013-46b7-b240-3cd5ca624323', '7f949ec4-b27e-4c6d-9d8c-814f3ed27b2b', '5ed6dfb6-e0b1-4685-a753-9407fda527c7', '5a077b27-5fa0-4064-96d8-2a428e742197', '10c7192c-6c11-4c06-90a8-981edfbbaa4a', '17af1482-070c-4dd0-87b1-4e81771c6110', '645f06af-e6ce-4d4c-ba31-ac5801524b91', '79750260-6423-439a-9e64-60be0846e72a', '0b1e70f3-fb33-4e26-95a0-bda6cb67c622', '23b7bacc-4363-4c88-b7e7-37290249aca6', '7c2e3ad7-34b3-4c98-8e9c-1d39dbef6dd8', 'ae2b929e-ab

In [61]:
import pandas as pd
df = pd.DataFrame({
    "ID": all_docs["ids"],
    "Document": all_docs["documents"],
    "Title": [doc.get("title", None) for doc in all_docs['metadatas']]
})
df.head()

,ID,Document,Title
0,98df6b87-62e0-47b9-80b7-935c6d0968cd,Glassblowing Technique and Research Q&A\nIntro...,None
1,37c7198d-dcf8-4dec-8bd7-e0cb3bcd09f0,Research Applications and Technology Integrati...,None
2,fe247e61-6868-4d33-9d1b-b9d73b89f021,Glassblowing Technique and Research Q&A\nIntro...,None
3,5602fbc0-2b8e-44c4-a24b-0a1e2e43617c,Research Applications and Technology Integrati...,None
4,fb5beb19-5dfd-4920-84db-b6e9e549fdf1,Note:\nThis site is no longer used and is in r...,Web service API functions - MoodleDocs


In [59]:
# Step 3: Check what metadata fields you have
# List all metadata dicts that contain a "title" field
docs_with_title = [doc for doc in all_docs['metadatas'] if "title" in doc]
for doc in docs_with_title:
    print(doc)

{'title': 'Web service API functions - MoodleDocs', 'page_label': '1', 'moddate': '2025-05-19T13:42:09+00:00', 'total_pages': 19, 'source': 'C:\\Users\\Theo Akbas\\Documents\\moodle_ai_assistant\\documents\\Web service API functions.pdf', 'page': 0, 'creationdate': '2025-05-19T13:42:09+00:00', 'producer': 'Skia/PDF m136', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/136.0.0.0 Safari/537.36'}
{'creationdate': '2025-05-19T13:42:09+00:00', 'title': 'Web service API functions - MoodleDocs', 'moddate': '2025-05-19T13:42:09+00:00', 'producer': 'Skia/PDF m136', 'total_pages': 19, 'page': 1, 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/136.0.0.0 Safari/537.36', 'page_label': '2', 'source': 'C:\\Users\\Theo Akbas\\Documents\\moodle_ai_assistant\\documents\\Web service API functions.pdf'}
{'title': 'Web service API functions - MoodleDocs', 'producer': 'Skia/PDF m136', 'creator': 'Mozilla

In [32]:
# Step 4: Test your remove function with a specific file
# Use the actual file path from your metadata
if all_docs['metadatas']:
    first_source = all_docs['metadatas'][1].get('source', 'no_source_found')
    print(f"Testing removal of: {first_source}")
    
    # Count documents before removal
    docs_before = len(vector_store.get()['ids'])
    
    # Try to remove documents
    success = remove_documents(vector_store, file_paths=[first_source])
    
    # Count documents after removal
    docs_after = len(vector_store.get()['ids'])
    
    print(f"Documents before: {docs_before}")
    print(f"Documents after: {docs_after}")
    print(f"Removal successful: {success}")

File path: C:\Users\Theo Akbas\Documents\moodle_ai_assistant\documents\Build a Retrieval Augmented Generation.pdf

2025-05-23   INFO   __main__:   Found a result: {'ids': ['a0f2406c-7f1d-477b-a9a7-24dac6ebd3e4', '4a4566e6-b470-4cc1-b899-127411b21d66', '0549ab31-1418-4479-84f5-8adf1f8d9b07', 'f0aa0529-73c3-4385-9e26-5bdbbe93907d', '55db04c9-0b73-4d86-b1f7-ec318b36e678', 'ffd3d36a-f932-4ba0-88a2-636ad1a1479d', '4229d588-2a93-40fe-965a-9891ca434368', '9ff4eae1-89de-49bd-8fda-0d18982d8a94', '69039e12-f864-4328-840b-9c57b446f1a6', '8ff7a299-ceeb-4c46-8f19-e3b174aff3c8', '941b7e25-1f6a-43d8-8a54-e67c018bb694', '8d9277ce-d01c-4faf-ba3c-1c629350f851', '0ee747b6-8c7e-49c4-ac0d-3a1c870f80a8', 'ab1bcf78-3fb4-49cd-bdb6-da41de72f188', 'ced955f0-f79a-4399-a215-29a923b35fc6', '8505ccb2-ac8b-41b9-ac26-4d0618e39d04', '4049ba36-4736-4497-b0b7-3f3bebd56e2b', 'f1343f07-e751-425f-826b-643e9fdef89e', '88a79949-2f10-4ff1-8568-a22eca04180e', '2f4c8ebe-50f3-45b0-95fd-c55787479962', '3226c564-4499-4b0f-bff1-557

Testing removal of: C:\Users\Theo Akbas\Documents\moodle_ai_assistant\documents\Build a Retrieval Augmented Generation.pdf
Documents before: 31
Documents after: 0
Removal successful: True


# Gradio

In [34]:
import gradio as gr

def filter_records(records, gender):
    return records[records["gender"] == gender]

demo = gr.Interface(
    filter_records,
    [
        gr.Dataframe(
            headers=["name", "age", "gender"],
            datatype=["str", "number", "str"],
            row_count=5,
            col_count=(3, "fixed"),
        ),
        gr.Dropdown(["M", "F", "O"]),
    ],
    "dataframe",
    description="Enter gender as 'M', 'F', or 'O' for other.",
)

if __name__ == "__main__":
    demo.launch()

* Running on local URL:  http://127.0.0.1:7861

To create a public link, set `share=True` in `launch()`.


## FFMPEG

In [ ]:
import subprocess
import os

def cut_video_ol(video_fp="C:\\Users\\Theo Akbas\\Desktop\\FredD_3rd_day.mp4", overwrite:bool=False, t:int=10, overlap:int=0, output_name:str="output", pad_last:bool=False, ):
    output_path = f"{output_name}.mp4"

    if os.path.exists(path=output_path):
        if not overwrite:
            user_input = input("WARNING - File already exists in directory. Do you want to overwrite (Y/n)? ")
            if user_input.lower() != "y":
                print("Operation cancelled.")
                return None

    vid_len = subprocess.run([
            "ffprobe",
            "-i", f"{file}",
            "-show_entries", "format=duration",
            "-v", "quiet",
            "-of", f"csv={"p=0"}" # csv is a writer. the option is "p=0" which disables displaying the section name at the begining and end
        ], capture_output=True, text=True).stdout.replace("\n", "")
    vid_len = int(float(vid_len))
    vid_chunks_num = vid_len // t
    print(f"Chunked video into {vid_chunks_num} chunks.\n")
    video_start_idx = [start_idx for start_idx in range(0, t-overlap, t-overlap)]
    video_end_idx = [end_idx + t for end_idx in video_start_idx]
    print(f"Video Start Times: {video_start_idx}\nVideo End Times: {video_end_idx}")
    vid_idx = 0
    
    

    result = subprocess.run([
        "ffmpeg",
        "-y",  # Force overwrite without prompting
        "-i", video_fp,
        "-c", "copy",
        "-t", str(t),
        f"{output_name}.mp4"
    ], capture_output=True, text=True)
    if result.returncode == 0:
        print("Successfully cut video.")
    else:
        print(f"Error occurred: {result.stderr}")
    return result

out_vid = cut_video_ol(overwrite=True, overlap=5)

Successfully cut video.


In [42]:
idx_x = [i for i in range(0, 11, 2)]
idx_y = [i + 3 for i in idx_x]

print(idx_x)
print(idx_y)

[0, 2, 4, 6, 8, 10]
[3, 5, 7, 9, 11, 13]


In [35]:
import subprocess
import os

file = "C:\\Users\\Theo Akbas\\Desktop\\FredD_3rd_day.mp4"

out = subprocess.run([
    "ffprobe",
    "-i", f"{file}",
    "-show_entries", "format=duration",
    "-v", "quiet",
    "-of", f"csv={"p=0"}" # csv is a writer. the option is "p=0" which disables displaying the section name at the begining and end
    ], capture_output=True, text=True)

if out.returncode != 0:
    print("Error: ", out.stderr, out)
else:
    print(int(float(out.stdout.replace("\n", ""))))

2112


In [40]:
123//4

30